# CS383: Data Science and Machine Learning
## Lecture 3 — Pandas and SQL Fundamentals

*Dr. Thitima Srivatanakul*

### Guiding question
**If SQL and Pandas can both answer the same question about a dataset, when would you reach for one over the other?**

### Learning objectives
By the end of this lecture, you should be able to:

- explain how a SQL table and a Pandas DataFrame relate to each other;
- write basic `SELECT`, `WHERE`, `GROUP BY`, `ORDER BY`, and `LIMIT` queries in SQL;
- perform the equivalent column selection, filtering, grouping, and ranking operations in Pandas;
- verify that a SQL query and a Pandas operation produce the same result;
- decide, for a simple question, whether SQL or Pandas is the more natural tool to reach for.

---

### Where this fits
Lecture 1 pulled real NYC 311 data straight from the API, and Lecture 2 built the Python/NumPy foundation underneath it. This lecture introduces the two tools you'll actually use to query and wrangle data like this day to day: SQL, for querying data that lives in a database, and Pandas, for working with it once it's in Python. Joins across multiple tables, real datetime handling, and messy real-world data cleaning are coming in Lecture 4 — today stays to a single, already-tidy table.

---
**Live in-class version.** Type along at each `__________` blank — everything else is filled in so class time stays on the new syntax, not on retyping boilerplate.

---

## Part 1 — DataFrames vs. Tables

A relational database stores data in **tables**: each row is a record, each column is a field. A Pandas **DataFrame** is the in-memory Python equivalent of a table — same basic shape, different tool.

| Database term | Pandas term |
|---|---|
| Table | DataFrame |
| Row / record | Row (indexed by `.index`) |
| Column / field | Column (a `Series`) |
| Query | Method chain / boolean mask |
| Primary key | Index (related idea, not always identical) |

### Setting up: the same data, in both forms

Below we pull the same live NYC 311 data style you saw in Lecture 1, load it into a Pandas DataFrame, and then write that DataFrame into an actual SQLite database file. By the end of this cell, the exact same data exists as both a DataFrame and a SQL table.

In [1]:
import sqlite3
import numpy as np
import pandas as pd
import requests

SOCRATA_URL = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

try:
    response = requests.get(
        SOCRATA_URL,
        params={
            "$limit": 8000,
            "$order": "created_date DESC",
            "$select": "unique_key,complaint_type,borough,created_date",
        },
        timeout=8,
    )
    response.raise_for_status()
    complaints_df = pd.DataFrame(response.json())
    live = True
except Exception:
    # Offline fallback, in case there is no internet connection in the room.
    rng = np.random.default_rng(383)
    n = 8000
    complaint_types = ["Noise - Residential", "Illegal Parking", "HEAT/HOT WATER",
                        "Blocked Driveway", "Street Condition", "Water System",
                        "PAINT/PLASTER", "Damaged Tree", "Sewer", "Rodent"]
    boroughs = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
    complaints_df = pd.DataFrame({
        "unique_key": np.arange(1, n + 1),
        "complaint_type": rng.choice(
            complaint_types, size=n,
            p=[0.18, 0.15, 0.14, 0.10, 0.10, 0.09, 0.08, 0.06, 0.05, 0.05],
        ),
        "borough": rng.choice(boroughs, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "created_date": pd.date_range("2026-01-01", periods=n, freq="min").astype(str),
    })
    live = False

print(f"{'Live' if live else 'Offline fallback'} data: {len(complaints_df):,} 311 records")

# Write the exact same data into a real SQLite database file.
conn = sqlite3.connect("nyc311_lab.db")
complaints_df.to_sql("complaints", conn, if_exists="replace", index=False)

print("Wrote table 'complaints' into nyc311_lab.db")

Live data: 8,000 311 records
Wrote table 'complaints' into nyc311_lab.db


### A first look, in both forms

In [2]:
complaints_df.__________()

,unique_key,complaint_type,borough,created_date
0,70024641,Illegal Parking,BRONX,2026-08-11T01:51:06.000
1,70021357,Illegal Parking,MANHATTAN,2026-08-11T01:50:41.000
2,70022914,Blocked Driveway,QUEENS,2026-08-11T01:50:01.000
3,70022802,Rodent,BRONX,2026-08-11T01:49:10.000
4,70021703,Illegal Posting,BRONX,2026-08-11T01:49:09.000


In [3]:
pd.read_sql_query("SELECT * FROM complaints LIMIT __________;", conn)

,unique_key,complaint_type,borough,created_date
0,70024641,Illegal Parking,BRONX,2026-08-11T01:51:06.000
1,70021357,Illegal Parking,MANHATTAN,2026-08-11T01:50:41.000
2,70022914,Blocked Driveway,QUEENS,2026-08-11T01:50:01.000
3,70022802,Rodent,BRONX,2026-08-11T01:49:10.000
4,70021703,Illegal Posting,BRONX,2026-08-11T01:49:09.000


### A quick but important detail: Series vs. DataFrame

You'll see the words "Series" and "DataFrame" constantly from here on — they're not interchangeable, and mixing them up is one of the most common sources of confusing errors in Pandas. Worth pinning down now, before we go further.

A **DataFrame** is the whole table — rows and columns, like `complaints_df` itself. A **Series** is a single column pulled out of that table: a 1-dimensional list of values with an index attached, but no column structure around it.

In [ ]:
print(type(complaints_df))
print(type(complaints_df["complaint_type"]))
print(type(complaints_df[[__________]]))

Same column, two different results depending on the brackets: a plain string (`df["col"]`) gives you back a **Series**; a list of column names — even a list of just one (`df[["col"]]`) — gives you back a **DataFrame**. This distinction comes up constantly. For example, `.value_counts()` (used throughout this lecture) returns a Series, which is exactly why later cells reach for `.index` to pull the labels back out of it, rather than treating the result like a column in a table.

### Checking the schema

Pandas describes a DataFrame's columns with `.dtypes`. SQLite has its own way to describe a table's schema: `PRAGMA table_info(table_name)`.

In [4]:
print(complaints_df.__________)

unique_key        object
complaint_type    object
borough           object
created_date      object
dtype: object


In [5]:
pd.read_sql_query("__________ table_info(complaints);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,unique_key,TEXT,0,None,0
1,1,complaint_type,TEXT,0,None,0
2,2,borough,TEXT,0,None,0
3,3,created_date,TEXT,0,None,0


Notice `created_date` is just plain text for now, in both forms. Real datetime handling — parsing it, computing durations from it — is Lecture 4's job. Today we leave it alone.

### Preview: the same question, two ways

Here's a taste of where this lecture is headed: the exact same question, answered once in SQL and once in Pandas.

In [9]:
sql = """
SELECT complaint_type, COUNT(*) AS n
FROM complaints
GROUP BY __________
ORDER BY n DESC
LIMIT 3;
"""
sql_result = pd.read_sql_query(sql, conn)
print(sql_result)

        complaint_type     n
0      Illegal Parking  1120
1  Noise - Residential   501
2    Water Maintenance   470


In [13]:
pandas_result = complaints_df["complaint_type"].__________().head(3)
print(pandas_result)

complaint_type
Illegal Parking        1120
Noise - Residential     501
Water Maintenance       470
Name: count, dtype: int64


In [17]:
sql_top3 = list(sql_result["complaint_type"])
pandas_top3 = list(pandas_result.index)

print(f"SQL top 3:    {sql_top3}")
print(f"Pandas top 3: {pandas_top3}")
print(f"Same answer? {sql_top3 __________ pandas_top3}")

SQL top 3:    ['Illegal Parking', 'Noise - Residential', 'Water Maintenance']
Pandas top 3: ['Illegal Parking', 'Noise - Residential', 'Water Maintenance']
Same answer? True


Index(['Illegal Parking', 'Noise - Residential', 'Water Maintenance'], dtype='object', name='complaint_type')

Same question, two completely different-looking pieces of code, same answer. The rest of this lecture builds up exactly how each of those pieces works, one at a time.

---

## Part 2 — SELECT: Choosing Columns

In [27]:
complaints_df[__________].head()

,complaint_type
0,Illegal Parking
1,Illegal Parking
2,Blocked Driveway
3,Rodent
4,Illegal Posting


In [19]:
pd.read_sql_query("SELECT __________ FROM complaints LIMIT 5;", conn)

,complaint_type,borough
0,Illegal Parking,BRONX
1,Illegal Parking,MANHATTAN
2,Blocked Driveway,QUEENS
3,Rodent,BRONX
4,Illegal Posting,BRONX


`df[["col1", "col2"]]` picks specific columns in Pandas — note the double brackets, since you're passing a *list* of column names. `SELECT col1, col2 FROM table` does the same thing in SQL. `SELECT *` (all columns) is just `df` on its own, or `df.head()` for a peek.

---

## Part 3 — WHERE: Filtering Rows

In [ ]:
pd.read_sql_query("SELECT * FROM complaints WHERE borough = '__________' LIMIT 5;", conn)

In [ ]:
complaints_df[complaints_df["borough"] __________ "BROOKLYN"].head()

### Combining conditions

In [ ]:
sql = """
SELECT * FROM complaints
WHERE borough = 'BROOKLYN' __________ complaint_type = 'Noise - Residential'
LIMIT 5;
"""
pd.read_sql_query(sql, conn)

In [ ]:
complaints_df[
    (complaints_df["borough"] == "BROOKLYN") __________ (complaints_df["complaint_type"] == "Noise - Residential")
].head()

### A real gotcha

In Pandas, combine conditions with `&` (and), `|` (or), and `~` (not) — **not** Python's `and`/`or`/`not` from Lecture 2. Those keywords only work on a single `True`/`False` value at a time; a Pandas condition like `complaints_df["borough"] == "BROOKLYN"` produces an entire column of `True`/`False` values, and `and`/`or` don't know what to do with that. You also need parentheses around each individual condition, because of how Python evaluates `&` and `|`.

### Checking they match

In [ ]:
sql_count = len(pd.read_sql_query(
    "SELECT * FROM complaints WHERE borough = 'BROOKLYN' AND complaint_type = 'Noise - Residential';", conn
))
pandas_count = len(complaints_df[
    (complaints_df["borough"] == "BROOKLYN") & (complaints_df["complaint_type"] == "Noise - Residential")
])

print(f"SQL row count:    {sql_count}")
print(f"Pandas row count: {pandas_count}")
print(f"Match: {sql_count __________ pandas_count}")

### A different trap: mixing up AND and OR

Getting `&`/`|` syntax right doesn't protect you from asking for the wrong logic. Say you want every complaint from Brooklyn *or* Queens:

In [ ]:
complaints_df[
    (complaints_df["borough"] == "BROOKLYN") & (complaints_df["borough"] == "QUEENS")
]

This is valid Pandas syntax and runs without any error — but it returns zero rows. Can you see why, before running it? (Hint: can a single complaint's `borough` value equal both `"BROOKLYN"` and `"QUEENS"` at the same time?)

In [ ]:
complaints_df[
    (complaints_df["borough"] == "BROOKLYN") __________ (complaints_df["borough"] == "QUEENS")
].shape

`&` requires every condition to be true for the same row at once — and a single complaint only has one borough, so demanding it equal two different boroughs simultaneously can never be satisfied. `|` asks for *either* condition, which is what was actually wanted. The `&`/`|` lesson earlier keeps you from a crash; this one keeps you from a silently wrong — but technically valid — answer, which is arguably more dangerous, since nothing tells you it happened.

### A cheap habit worth building

In [ ]:
brooklyn_only = complaints_df[complaints_df["borough"] == "BROOKLYN"]
brooklyn_only["borough"].__________()

Anytime you filter a DataFrame, a quick `value_counts()` on the column you just filtered by is a cheap way to confirm the filter actually did what you think. If this ever shows anything besides a single `BROOKLYN` row, the filter is wrong — far cheaper to catch it here than three steps later.

---

## Part 4 — GROUP BY, Aggregation, and Ranking

In [ ]:
sql = """
SELECT borough, COUNT(*) AS complaint_count
FROM complaints
GROUP BY __________
ORDER BY complaint_count DESC;
"""
pd.read_sql_query(sql, conn)

In [ ]:
complaints_df.groupby("__________").size().sort_values(ascending=False)

`GROUP BY col` plus `COUNT(*)` in SQL maps to `df.groupby("col").size()` in Pandas — both count how many rows fall into each group. `.value_counts()` (which you used in Part 1) is a shortcut for exactly this pattern when you're grouping and counting by a single column.

### Top-N ranking with ORDER BY and LIMIT

Remember Lecture 1's "Top 10 NYC 311 Complaint Types" bar chart? It was built from `.value_counts().head(10)`. Here's the same ranked list, this time straight from a SQL query.

In [ ]:
sql = """
SELECT complaint_type, COUNT(*) AS complaint_count
FROM complaints
GROUP BY complaint_type
ORDER BY complaint_count DESC
LIMIT __________;
"""
pd.read_sql_query(sql, conn)

In [ ]:
complaints_df["complaint_type"].value_counts().__________(5)

`ORDER BY col DESC` sorts largest-to-smallest, matching `.sort_values(ascending=False)`; `LIMIT n` keeps only the top rows, matching `.head(n)`. `.value_counts()` conveniently does the sorting for you automatically.

---

**Exercises for this lecture** (Lab, Exit Ticket, Optional Challenge) live in a separate notebook: `lect03_pandas_sql_fundamentals_exercise.ipynb`.

---

## Part 5 — Where Pandas Pulls Ahead

Every Pandas example so far has been a direct mirror of something SQL just did. That's true for the basics — but a lot of everyday Pandas work doesn't have as clean a SQL equivalent, or is just noticeably faster to write. A few examples, using nothing beyond what you already know.

### Percentages, not just counts

In [ ]:
complaints_df["borough"].value_counts(__________=True).round(3) * 100

`value_counts(normalize=True)` gives you proportions instead of raw counts in a single call. The SQL equivalent needs a subquery to divide each group's count by the grand total — a noticeably bigger jump in difficulty for the same question.

### Chaining several steps in one expression

In [ ]:
(
    complaints_df[complaints_df["borough"] __________ "STATEN ISLAND"]
    .groupby("complaint_type")
    .size()
    .sort_values(ascending=False)
    .head(5)
)

Filter, group, count, sort, and trim to the top 5 — five operations, one continuous, readable expression. SQL can express this too (with a `WHERE`, a `GROUP BY`, and an `ORDER BY`), but Pandas method-chaining tends to read more like a sentence: 'take this, then do this, then do this.'

### A two-way count table

In [ ]:
pd.__________(complaints_df["borough"], complaints_df["complaint_type"])

`pd.crosstab()` builds a full borough-by-complaint-type count table in one line. This is the kind of thing SQL is genuinely awkward at — a table like this needs a `PIVOT`, which standard SQLite doesn't even support natively; you'd be stuck hand-writing a `CASE WHEN` for every single complaint type. Sometimes the right tool really is Pandas, not because SQL *can't*, but because it makes you work much harder for the same answer.

### A convenience worth a caution: `.apply()`

Pandas has an `.apply()` method that runs any custom function against a column — genuinely convenient for logic that doesn't have a ready-made vectorized equivalent.

In [ ]:
import time

start = time.time()
_ = complaints_df["complaint_type"].apply(lambda x: x.upper())
apply_time = time.time() - start

start = time.time()
_ = complaints_df["complaint_type"].str.__________()
vectorized_time = time.time() - start

print(f".apply():      {apply_time:.4f} seconds")
print(f".str.upper():  {vectorized_time:.4f} seconds")
print(f"The vectorized .str version was about {apply_time / vectorized_time:,.1f}x faster")

`.apply()` is not vectorized just because it's a one-liner — under the hood it calls your Python function once per row, exactly like the loop you timed back in Lecture 2 (and exactly like `np.vectorize()`, if that sounds familiar). Reach for a real vectorized option first — elementwise math, `.str` methods, `np.where()` — and save `.apply()` for logic that's genuinely too custom to express any other way.

---

## Part 6 — SQL ↔ Pandas Cheat Sheet

| Task | SQL | Pandas |
|---|---|---|
| Choose columns | `SELECT col1, col2 FROM table` | `df[["col1", "col2"]]` |
| Filter rows | `WHERE condition` | `df[condition]` |
| Combine conditions | `AND` / `OR` | `&` / `\|` (parentheses around each condition) |
| Count per group | `GROUP BY col` + `COUNT(*)` | `df.groupby("col").size()` or `df["col"].value_counts()` |
| Sort | `ORDER BY col DESC` | `.sort_values(ascending=False)` |
| Top N rows | `LIMIT n` | `.head(n)` |
| Preview rows | `LIMIT 5` | `.head()` |
| Table/DataFrame schema | `PRAGMA table_info(table)` | `.dtypes` |

---

## Part 7 — Key Terms

- **Table**: a structured collection of rows and columns in a database.
- **Row / record**: one entry in a table (or DataFrame).
- **Column / field**: one attribute shared across all rows; a single column in a DataFrame is a `Series`.
- **Query**: a request for data from a database, written in SQL.
- **Schema**: the structure of a table — its column names and types.
- **Boolean mask**: an array of `True`/`False` values used to filter a DataFrame (e.g., `df["borough"] == "BROOKLYN"`).
- **Aggregate function**: a function that reduces many rows to one summary value per group (`COUNT`, and later `AVG`, `SUM`, etc.).
- **GROUP BY / `.groupby()`**: splitting rows into groups based on shared column values before aggregating.
- **Connection / cursor**: the object Python uses to communicate with a database (`sqlite3.connect(...)`).